In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import sys
import os
sys.path.append(os.path.abspath("../Code"))
import preproc

df = pd.read_csv('../Data/processed/tcga_simple_train_preprocessed_0.csv')
df["text"] = preproc.preprocesamiento(df["text"],4)

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['t'], test_size=0.2, random_state=23)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=33)

KeyboardInterrupt: 

In [ ]:
from transformers import BertTokenizer
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight


tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
# tokenizer = BertTokenizer.from_pretrained('dmis-lab/biobert-v1.1')

label_mapping = {"T1": 0, "T2": 1, "T3": 2, "T4": 3}

train_ds = Dataset.from_dict({"text": X_train.tolist(), "labels": [label_mapping[label] for label in y_train.tolist()]})
test_ds = Dataset.from_dict({"text": X_test.tolist(), "labels": [label_mapping[label] for label in y_test.tolist()]})
val_ds = Dataset.from_dict({"text": X_val.tolist(), "labels": [label_mapping[label] for label in y_val.tolist()]})

# Tokenize with batched=True for significant speedup
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

X_train_tkn = train_ds.map(tokenize_function, batched=True)
X_val_tkn = val_ds.map(tokenize_function, batched=True)
X_test_tkn = test_ds.map(tokenize_function, batched=True)


le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
weights = compute_class_weight(
    class_weight="balanced", 
    classes=np.unique(y_train_encoded),
    y=y_train_encoded
)

Map:   0%|          | 0/3300 [00:00<?, ? examples/s]

Map:   0%|          | 0/826 [00:00<?, ? examples/s]

Map:   0%|          | 0/1032 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForSequenceClassification
import torch

model_name = "bert-base-uncased"
# model_name = 'dmis-lab/biobert-v1.1'
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)
model.to("cuda" if torch.cuda.is_available() else "cpu")
model.loss_function = torch.nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float).to(model.device))

"""
if model_name == 'dmis-lab/biobert-v1.1':
    for param in model.bert.parameters():
        param.requires_grad = False
"""

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dmis-lab/biobert-v1.1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


"\nif model_name == 'dmis-lab/biobert-v1.1':\n    for param in model.bert.parameters():\n        param.requires_grad = False\n"

In [ ]:
from transformers import TrainingArguments


training_args = TrainingArguments(
    output_dir="../Models/bert-classification",
    eval_strategy="epoch",           # Evaluate at the end of each epoch
    save_strategy="epoch",           # Added: Save at the end of each epoch to match eval
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=20,
    weight_decay=0.05,
    save_total_limit=2,
    load_best_model_at_end=True,     # Now works because strategies match
    metric_for_best_model="loss",    # Tell the trainer to track 'loss' (or 'accuracy')
    logging_dir="../Models/bert-classification/logs",
    logging_steps=100,
    fp16=True                        # Mixed precision (requires a compatible GPU)
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
from sklearn.metrics import f1_score
from transformers import DataCollatorWithPadding, Trainer, EarlyStoppingCallback

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    macro_f1 = f1_score(labels, preds, average='macro')
    return {'macro_f1': macro_f1}

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,                        # Pre-trained BERT model
    args=training_args,                 # Training arguments
    train_dataset=X_train_tkn,
    eval_dataset=X_val_tkn,
    processing_class=tokenizer,
    data_collator=data_collator,        # Efficient batching
    compute_metrics=compute_metrics         # Evaluation metrics
)

trainer.add_callback(EarlyStoppingCallback(early_stopping_patience=3))

# Start training
trainer.train()

Epoch,Training Loss,Validation Loss,Macro F1
1,1.103100,1.098953,0.505616
2,1.045536,1.096009,0.510604
3,0.790336,0.853123,0.614584
4,0.566143,0.620841,0.747522
5,0.419106,0.671975,0.760264
6,0.351168,0.735976,0.750418
7,0.273128,0.962135,0.731306


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1449, training_loss=0.6652620893416527, metrics={'train_runtime': 668.7732, 'train_samples_per_second': 98.688, 'train_steps_per_second': 6.19, 'total_flos': 6077974520217600.0, 'train_loss': 0.6652620893416527, 'epoch': 7.0})

In [ ]:
from sklearn.metrics import classification_report

predictions = trainer.predict(X_test_tkn)
predicted_labels = predictions.predictions.argmax(axis=-1)

y_test_encoded = le.fit_transform(y_test)

print(classification_report(
    y_test_encoded, 
    predicted_labels, 
    target_names=le.classes_
))

              precision    recall  f1-score   support

          T1       0.79      0.70      0.74       247
          T2       0.70      0.83      0.76       354
          T3       0.88      0.77      0.82       310
          T4       0.76      0.74      0.75       121

    accuracy                           0.77      1032
   macro avg       0.78      0.76      0.77      1032
weighted avg       0.78      0.77      0.77      1032

